CNN

In [ ]:
!pip install gensim
!pip install tensorflow
from google.colab import files
uploaded=files.upload()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 39.4 MB/s eta 0:00:00


Saving test.csv to test.csv


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Conv1D
from tensorflow.keras.layers import MaxPooling1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten, Input, Dropout
from sklearn.model_selection import train_test_split
from gensim.parsing.preprocessing import preprocess_string
from nltk import sent_tokenize

In [ ]:
df = pd.read_csv('data_mr.csv').iloc[7000:18000]

In [ ]:
df.drop_duplicates(inplace=True)


In [ ]:
df

,text,label
7000,Bad. Personal opinion? The folks who made it? ...,0
7001,I have to preface this by saying that I LOVE w...,0
7002,"Awful, waste of time. There is no camp or tras...",0
7003,<br /><br />In the process of boring you with ...,0
7004,I could not believe it. This film was a total ...,0
...,...,...
17995,Bwana Devil is reputedly the first major studi...,1
17996,Bravestarr was released in 1987 by the now def...,1
17997,Man! I remember this show with nostalgic... I ...,1
17998,"""Brotherhood of Satan"" is one of the most unde...",1


In [ ]:
df['text'] = df['text'].apply(preprocess_string)
df['text']=df['text'].apply(lambda x: ' '.join(x))


In [ ]:
y=df['label']

In [ ]:
X=df['text']

In [ ]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=7)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_words = 10000


In [ ]:
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
max_len = 200

X_train_pad = pad_sequences(X_train_seq,
                            maxlen=max_len,
                            padding='post')

X_test_pad = pad_sequences(X_test_seq,
                           maxlen=max_len,
                           padding='post')

In [ ]:
model=Sequential()
model.add(Embedding(input_dim=max_words,output_dim=128,input_length=max_len))
model.add(Conv1D(filters=128,kernel_size=5,activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 14s 19ms/step - accuracy: 0.7150 - loss: 0.5118 - val_accuracy: 0.8758 - val_loss: 0.3080
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9360 - loss: 0.1880 - val_accuracy: 0.8792 - val_loss: 0.3076
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9782 - loss: 0.0679 - val_accuracy: 0.8632 - val_loss: 0.4419
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9903 - loss: 0.0322 - val_accuracy: 0.8718 - val_loss: 0.6508
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9943 - loss: 0.0179 - val_accuracy: 0.8712 - val_loss: 0.7195
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9974 - loss: 0.0090 - val_accuracy: 0.8684 - val_loss: 0.9148
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9989 - loss: 0.0042 - val_accuracy: 0.8593 - val_loss: 1.2192
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9980 - loss: 0.0050 - val_accuracy: 

In [ ]:
model.evaluate(X_test_pad,y_test)

69/69 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.8815 - loss: 0.8679


[0.8679024577140808, 0.8814949989318848]

In [ ]:
from sklearn.metrics import f1_score

y_pred = (model.predict(X_test_pad) > 0.5).astype(int)
print(f1_score(y_test, y_pred))

69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
0.8841354723707665
